# Phase 6 — partial fine-tuning with learned sentence selection and tuned decoding


In [1]:
%pip install -q huggingface_hub tokenizers scikit-learn tqdm rouge-score


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, math, random, gc, tempfile, zipfile, uuid, re, hashlib
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfFileSystem
from rouge_score import rouge_scorer
import time
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
DATA_REPO = 'Exploration-Lab/CS-781-Capstone'
BUCKET_ROOT = 'hf://buckets/A-240031/legal-lm-phase1-bucket'
PHASE1_TAG = 'encoder_decoder_mixed'
MODEL_PATH = 'checkpoints/encoder_decoder_mixed/model.pt'
TOKENIZER_PATH = 'tokenizer_encoder_decoder/tokenizer.json'
MODEL_CONFIG = {'dim': 640, 'encoder_layers': 6, 'decoder_layers': 6, 'n_heads': 16, 'n_kv_heads': 4, 'ff_dim': 1728, 'max_encoder_len': 512, 'max_decoder_len': 256}
MAX_SEQ_LEN = 512  # source encoder budget; decoder has its own budget
USE_GRADIENT_CHECKPOINTING = True
UNFREEZE_ENCODER_BLOCKS = 2  # full attention, FFN and norms in the last two blocks
UNFREEZE_DECODER_BLOCKS = 2  # includes self-attention and cross-attention
TRAIN_FINAL_NORMS = True
MAX_NEW_TOKENS = 256  # includes EOS; separate decoder budget
SOURCE_SPANS = 8  # retained baseline compatibility; learned selector does not use spans
VAL_FRACTION = 0.05
VAL_MAX_DOCS = 100  # fixed held-out subset used for generation-based selection; None = full dev
LOG_EVERY_DOCS = 1000
USE_AMP = DEVICE == 'cuda'
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16

def autocast_context():
    return torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP)

DATA_PREFIX = 'Phase_6_Summ'
SUMMARY_KEY = 'summary'  # set explicitly if the dataset uses a different field
SKIP_EMPTY_SUMMARIES = True  # null/blank training targets cannot supervise generation
WARMUP_RATIO = 0.05
WARMUP_START_FACTOR = 0.1
LR_FACTOR = 0.5
LR_BAD_CHECKS = 2
LR_COOLDOWN_CHECKS = 1
LR_MIN = 1e-6
LR_LOSS_THRESHOLD = 0.001  # require 0.1% relative dev-loss improvement
VAL_LOSS_CHECKS_PER_EPOCH = 4
assert 0 < MAX_SEQ_LEN <= MODEL_CONFIG['max_encoder_len']
assert 1 < MAX_NEW_TOKENS <= MODEL_CONFIG['max_decoder_len']
assert SOURCE_SPANS > 0
assert 0 <= UNFREEZE_ENCODER_BLOCKS <= MODEL_CONFIG['encoder_layers']
assert 0 <= UNFREEZE_DECODER_BLOCKS <= MODEL_CONFIG['decoder_layers']
assert UNFREEZE_ENCODER_BLOCKS + UNFREEZE_DECODER_BLOCKS > 0
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
LR = 2e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
NUM_EPOCHS = 5  # retained from the reference; lower this to cap training time
PATIENCE = 10
MIN_DELTA = 0.001
UPLOAD_CHECKPOINTS = True
TECHNIQUE = 'partial_ft_sentence_select'
TRAIN_FRACTION = 1.0  # use all training documents after holding out validation

# All loading/saving locations are controlled here.
# Paths below are relative to BUCKET_ROOT; an absolute hf:// path also works.
SAVE_TAG = None  # None generates a descriptive tag; or enter your own run tag.
SUBMISSION_ZIP_NAME = None  # None uses SAVE_TAG + '.zip'; or enter a filename.
CHECKPOINT_PREFIX = 'checkpoints'
RESUME = False  # True resumes SAVE_TAG; set SAVE_TAG to the existing latest tag.
WORK_ROOT = Path('/kaggle/working/summ_runs') if Path('/kaggle/working').exists() else Path.cwd() / 'summ_runs'

if RESUME and SAVE_TAG is None:
    raise ValueError('To resume, set SAVE_TAG to the existing run tag.')
if SAVE_TAG is None:
    SAVE_TAG = f'summ_{PHASE1_TAG}_{TECHNIQUE}_enc{UNFREEZE_ENCODER_BLOCKS}_dec{UNFREEZE_DECODER_BLOCKS}_len{MAX_SEQ_LEN}_summary{MAX_NEW_TOKENS}_selected_warmup_plateau'
if not isinstance(SAVE_TAG, str) or not re.fullmatch(r'[A-Za-z0-9_.-]+', SAVE_TAG) or SAVE_TAG in {'.', '..'}:
    raise ValueError('SAVE_TAG must be a single name containing letters, numbers, _, - or .')
BEST_TAG = SAVE_TAG + '_best'
if SUBMISSION_ZIP_NAME is None:
    SUBMISSION_ZIP_NAME = SAVE_TAG + '.zip'
if Path(SUBMISSION_ZIP_NAME).name != SUBMISSION_ZIP_NAME or not SUBMISSION_ZIP_NAME.endswith('.zip'):
    raise ValueError('SUBMISSION_ZIP_NAME must be a filename ending in .zip.')
WORK_DIR = WORK_ROOT / SAVE_TAG
WORK_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF')
    except Exception:
        pass
fs = HfFileSystem(token=HF_TOKEN)

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(SEED)
print('Device:', DEVICE)
def bucket_path(path):
    if path.startswith('hf://'):
        return path
    return BUCKET_ROOT.rstrip('/') + '/' + path.lstrip('/')

def checkpoint_remote_path(tag):
    return bucket_path(f"{CHECKPOINT_PREFIX.strip('/')}/{tag}/checkpoint.pt")

def download_bucket_file(remote_path):
    # Identity-specific cache directory; refresh the bytes on each call.
    remote = bucket_path(remote_path)
    cache_key = hashlib.sha256(remote.encode()).hexdigest()[:20]
    local = WORK_ROOT / '_inputs' / cache_key / remote.rsplit('/', 1)[-1]
    local.parent.mkdir(parents=True, exist_ok=True)
    temporary = local.with_name(local.name + '.download')
    fs.get_file(remote, str(temporary))
    os.replace(temporary, local)
    return local

def assert_run_destination():
    if RESUME:
        return
    for tag in [SAVE_TAG, BEST_TAG]:
        local = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
        if local.exists() or (UPLOAD_CHECKPOINTS and fs.exists(checkpoint_remote_path(tag))):
            raise FileExistsError(
                f'Checkpoint already exists for {tag}. Choose a new SAVE_TAG, '
                'or set RESUME=True with the original latest SAVE_TAG.'
            )
    manifest = experiment_metadata('SUMM')
    manifest.update(save_tag=SAVE_TAG, best_tag=BEST_TAG, submission_zip=SUBMISSION_ZIP_NAME)
    with open(WORK_DIR / 'run_config.json', 'w') as f:
        json.dump(manifest, f, indent=2)

print('Phase-1 model:', bucket_path(MODEL_PATH))
print('Tokenizer:', bucket_path(TOKENIZER_PATH))
print('Latest checkpoint:', checkpoint_remote_path(SAVE_TAG))
print('Best checkpoint:', checkpoint_remote_path(BEST_TAG))
print('Submission ZIP:', WORK_DIR / SUBMISSION_ZIP_NAME)

print('Encoder / decoder token budgets:', MAX_SEQ_LEN, MAX_NEW_TOKENS)


Device: cuda
Phase-1 model: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/encoder_decoder_mixed/model.pt
Tokenizer: hf://buckets/A-240031/legal-lm-phase1-bucket/tokenizer_encoder_decoder/tokenizer.json
Latest checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/summ_encoder_decoder_mixed_partial_ft_sentence_select_enc2_dec2_len512_summary256_selected_warmup_plateau/checkpoint.pt
Best checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/summ_encoder_decoder_mixed_partial_ft_sentence_select_enc2_dec2_len512_summary256_selected_warmup_plateau_best/checkpoint.pt
Submission ZIP: /kaggle/working/summ_runs/summ_encoder_decoder_mixed_partial_ft_sentence_select_enc2_dec2_len512_summary256_selected_warmup_plateau/summ_encoder_decoder_mixed_partial_ft_sentence_select_enc2_dec2_len512_summary256_selected_warmup_plateau.zip
Encoder / decoder token budgets: 512 256


In [3]:
local_tokenizer = download_bucket_file(TOKENIZER_PATH)
tokenizer = Tokenizer.from_file(str(local_tokenizer))
tokenizer.no_padding()
tokenizer.no_truncation()
VOCAB_SIZE = tokenizer.get_vocab_size()
EMPTY_ID = tokenizer.token_to_id('<|eos|>')
PAD_ID = tokenizer.token_to_id('<|pad|>')
if EMPTY_ID is None:
    raise ValueError('Expected the Phase-1 <|eos|> token.')
if PAD_ID is None: PAD_ID = EMPTY_ID


EOS_ID = EMPTY_ID
BOS_ID = tokenizer.token_to_id('<|bos|>')
# Use the existing EOS token as decoder start if Phase 1 has no BOS token.
# The same start token is used in teacher forcing and generation; no vocabulary changes.
DECODER_START_ID = EOS_ID if BOS_ID is None else BOS_ID
PASSAGE_GAP_IDS = tokenizer.encode('\n[...]\n', add_special_tokens=False).ids
SOURCE_BUDGET = MAX_SEQ_LEN
assert SOURCE_BUDGET >= SOURCE_SPANS
print('Source budget:', SOURCE_BUDGET, '| target budget:', MAX_NEW_TOKENS, '| decoder start ID:', DECODER_START_ID)


(…)tokenizer_encoder_decoder/tokenizer.json:   0%|          | 0.00/2.39M [00:00<?, ?B/s]

Source budget: 512 | target budget: 256 | decoder start ID: 2


In [4]:
def download_json(filename):
    path = hf_hub_download(repo_id=DATA_REPO, filename=filename, repo_type='dataset', token=HF_TOKEN)
    with open(path, encoding='utf-8') as f: return json.load(f)

def normalize_documents(raw, labelled):
    if isinstance(raw, list): docs = raw
    elif isinstance(raw, dict) and 'id' in raw and 'document' in raw: docs = [raw]
    elif isinstance(raw, dict):
        docs = []
        for doc_id, value in raw.items():
            if not isinstance(value, dict): raise ValueError('Expected ID -> document object mapping')
            doc = dict(value)
            doc.setdefault('id', doc_id)
            if doc['id'] != doc_id: raise ValueError('Document ID mismatch')
            docs.append(doc)
    else: raise ValueError('Expected list of document objects')
    result, excluded = [], []
    seen_ids = set()
    for doc in docs:
        doc_id, text = doc['id'], doc['document']
        if not isinstance(doc_id, str) or not isinstance(text, str) or not text.strip():
            raise ValueError('Expected string ID and nonempty document text')
        if doc_id in seen_ids: raise ValueError(f'Duplicate document ID: {doc_id}')
        seen_ids.add(doc_id)
        row = dict(id=doc_id, document=text)
        if labelled:
            if SUMMARY_KEY not in doc:
                raise ValueError(f'{doc_id}: summary field {SUMMARY_KEY!r} is absent. Available fields: {list(doc)}. Set SUMMARY_KEY to the actual reference field; missing fields are not silently skipped.')
            target = doc[SUMMARY_KEY]
            if target is None or (isinstance(target, str) and not target.strip()):
                if not SKIP_EMPTY_SUMMARIES:
                    raise ValueError(f'{doc_id}: null or blank reference summary')
                excluded.append(dict(id=doc_id, reason='null' if target is None else 'blank'))
                continue
            if not isinstance(target, str):
                raise ValueError(f'{doc_id}: {SUMMARY_KEY!r} must contain a string; got {type(target).__name__}. Inspect this record rather than coercing the target.')
            row['summary'] = target
        result.append(row)
    if labelled:
        print(f'Training input records: {len(docs):,} | usable references: {len(result):,} | excluded null/blank summaries: {len(excluded):,}')
        if excluded: print('Excluded IDs (first 20):', [r['id'] for r in excluded[:20]])
        with open(WORK_DIR / 'excluded_training_summaries.json', 'w') as f:
            json.dump(excluded, f, indent=2)
    if not result: raise ValueError('No usable records remain; inspect the dataset reference field')
    return result

all_train = normalize_documents(download_json(f'{DATA_PREFIX}/Train/train.json'), labelled=True)
# Exact duplicate document texts stay together, even if IDs differ.
groups = {}
for row in all_train:
    digest = hashlib.sha256(' '.join(row['document'].split()).encode()).hexdigest()
    groups.setdefault(digest, []).append(row)
keys = sorted(groups)
random.Random(SEED).shuffle(keys)
if len(keys) < 2: raise ValueError('At least two distinct documents are required')
assert 0 < VAL_FRACTION < 1 and 0 < TRAIN_FRACTION <= 1
n_dev = min(len(keys)-1, max(1, round(len(keys)*VAL_FRACTION)))
dev_keys, train_keys = keys[:n_dev], keys[n_dev:]
train_keys = train_keys[:max(1, round(len(train_keys)*TRAIN_FRACTION))]
data = {'train': [d for k in train_keys for d in groups[k]],
        'dev': [d for k in dev_keys for d in groups[k]]}
# Stable fixed subset, independent of generated outputs.
if VAL_MAX_DOCS is not None: data['dev'] = data['dev'][:VAL_MAX_DOCS]
assert data['dev'] and not ({d['id'] for d in data['train']} & {d['id'] for d in data['dev']})
DATA_FINGERPRINT = hashlib.sha256(json.dumps(all_train, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
with open(WORK_DIR / 'split_ids.json', 'w') as f:
    json.dump({s: [d['id'] for d in rows] for s, rows in data.items()}, f, indent=2)
print('Public training records:', len(all_train), '| train:', len(data['train']), '| validation used:', len(data['dev']))
del all_train, groups


Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

Training input records: 7,030 | usable references: 7,028 | excluded null/blank summaries: 2
Excluded IDs (first 20): ['299', '4799']
Public training records: 7028 | train: 6676 | validation used: 100


In [5]:
import numpy as np
import sklearn
from scipy import sparse
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier

SELECTOR_CONFIG = dict(version='sentence_overlap_supervised_v1', epochs=2, features=2**18,
                       max_unit_tokens=256, redundancy_penalty=0.25)
vectorizer = HashingVectorizer(n_features=SELECTOR_CONFIG['features'], alternate_sign=False,
                              ngram_range=(1,2), norm='l2')

def lexical_set(text):
    return set(re.findall(r'\b\w+\b',text.lower()))

def sentence_units(text):
    # Rule-based sentence/paragraph splitting; unusually long sentences become
    # bounded token fragments rather than being silently discarded.
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9])|\n\s*\n',text)
    units = []
    for part in parts:
        ids = tokenizer.encode(part.strip(),add_special_tokens=False).ids
        for start in range(0,len(ids),SELECTOR_CONFIG['max_unit_tokens']):
            chunk = ids[start:start+SELECTOR_CONFIG['max_unit_tokens']]
            decoded = tokenizer.decode(chunk,skip_special_tokens=True)
            units.append(dict(text=decoded,ids=chunk,words=lexical_set(decoded)))
    return units

def selector_features(units):
    n = max(1,len(units)-1)
    cues = [r'\b(held|hold|conclude|conclusion)\b',r'\b(appeal|petition|dismissed|allowed)\b',
            r'\b(issue|question|whether)\b',r'\b(because|therefore|reason|accordingly)\b']
    numeric = [[i/n,1-i/n,float(i==0),float(i==len(units)-1),len(u['ids'])/256,
                *[float(bool(re.search(p,u['text'],re.I))) for p in cues]] for i,u in enumerate(units)]
    return sparse.hstack([vectorizer.transform([u['text'] for u in units]),
                          sparse.csr_matrix(np.asarray(numeric,dtype=np.float64))],format='csr')

def oracle_sentence_labels(units,reference):
    # Training labels only. No reference is accepted by the inference selector.
    ref = lexical_set(reference)
    labels = np.zeros(len(units),dtype=np.int64)
    covered,used = set(),0
    while True:
        candidates = []
        for i,u in enumerate(units):
            cost = len(u['ids']) + (len(PASSAGE_GAP_IDS) if used else 0)
            if labels[i] or used+cost>SOURCE_BUDGET: continue
            gain = len((u['words'] & ref)-covered)
            candidates.append((gain/max(1,len(u['ids']))**0.5,-i,i,cost))
        if not candidates: break
        gain,_,i,cost=max(candidates)
        if gain<=0: break
        labels[i]=1;used+=cost;covered.update(units[i]['words'] & ref)
    return labels

selector = SGDClassifier(loss='log_loss',alpha=1e-5,random_state=SEED,average=True)
selector_updates = 0
for epoch in range(SELECTOR_CONFIG['epochs']):
    order = list(range(len(data['train'])))
    random.Random(SEED+epoch).shuffle(order)
    for index in tqdm(order,desc=f'Sentence selector epoch {epoch+1} (CPU)'):
        doc = data['train'][index]
        units = sentence_units(doc['document'])
        if not units: continue
        labels = oracle_sentence_labels(units,doc['summary'])
        positive = int(labels.sum())
        # Bounded weighting prevents rare positives being overwhelmed by negatives.
        weights = np.where(labels==1,min(20.,max(1.,(len(labels)-positive)/max(1,positive))),1.)
        selector.partial_fit(selector_features(units),labels,classes=np.array([0,1]),sample_weight=weights)
        selector_updates += 1
if not selector_updates: raise ValueError('No sentences available to train the selector')
SELECTOR_HASH = hashlib.sha256(selector.coef_.tobytes()+selector.intercept_.tobytes()).hexdigest()
# Also embedded in every training checkpoint; only this public training split fits it.
SELECTOR_PAYLOAD = dict(model=selector,config=SELECTOR_CONFIG,sha256=SELECTOR_HASH,
                        sklearn_version=sklearn.__version__)
torch.save(SELECTOR_PAYLOAD,WORK_DIR/'sentence_selector.pt')

def select_document_tokens(text):
    units = sentence_units(text)
    if not units: return [EOS_ID]
    scores = selector.predict_proba(selector_features(units))[:,1]
    chosen,used,covered = [],0,set()
    while True:
        candidates=[]
        for i,u in enumerate(units):
            cost=len(u['ids'])+(len(PASSAGE_GAP_IDS) if chosen else 0)
            if i in chosen or used+cost>SOURCE_BUDGET: continue
            overlap=len(u['words'] & covered)/max(1,len(u['words']))
            value=float(scores[i])-SELECTOR_CONFIG['redundancy_penalty']*overlap
            candidates.append((value,-i,i,cost))
        if not candidates: break
        _,_,i,cost=max(candidates)
        chosen.append(i);used+=cost;covered.update(units[i]['words'])
    result=[]
    for i in sorted(chosen):
        if result:result.extend(PASSAGE_GAP_IDS)
        result.extend(units[i]['ids'])
    assert len(result)<=SOURCE_BUDGET
    return result or [EOS_ID]

# Limited qualitative audit: only training documents, no private test labels.
with open(WORK_DIR/'source_selection_examples.json','w') as f:
    json.dump([dict(id=d['id'],selected=tokenizer.decode(select_document_tokens(d['document'])))
               for d in data['train'][:10]],f,indent=2,ensure_ascii=False)

class SummaryDataset(Dataset):
    def __init__(self, documents, split):
        self.rows, self.split = [], split
        source_lengths, target_lengths = [], []
        for doc in tqdm(documents, desc=f'Tokenizing {split}'):
            source = tokenizer.encode(doc['document'], add_special_tokens=False).ids
            source_ids = select_document_tokens(doc['document'])
            row = dict(doc_id=doc['id'], source_ids=source_ids)
            source_lengths.append(len(source))
            if 'summary' in doc:
                target = tokenizer.encode(doc['summary'], add_special_tokens=False).ids
                if not target: raise ValueError(f"Empty tokenized summary: {doc['id']}")
                target_lengths.append(len(target) + 1)
                row.update(target=target[:MAX_NEW_TOKENS-1] + [EOS_ID], reference=doc['summary'])
            self.rows.append(row)
        for name, lengths in [('document', source_lengths), ('summary including EOS', target_lengths)]:
            if lengths:
                print(split, name, 'tokens p50/p90/p99/max:', torch.quantile(torch.tensor(lengths).float(), torch.tensor([.5,.9,.99,1.])).tolist())
        print(split, 'documents needing source selection:', sum(n > SOURCE_BUDGET for n in source_lengths))
        if target_lengths: print(split, 'reference summaries truncated for training/loss:', sum(n > MAX_NEW_TOKENS for n in target_lengths), '/', len(target_lengths))
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collate_summary(rows):
    source_width = max(len(r['source_ids']) for r in rows)
    target_width = max(len(r['target']) for r in rows)
    assert source_width <= MAX_SEQ_LEN and target_width <= MAX_NEW_TOKENS
    encoder_ids = torch.full((len(rows), source_width), PAD_ID, dtype=torch.long)
    encoder_mask = torch.zeros_like(encoder_ids, dtype=torch.bool)
    decoder_ids = torch.full((len(rows), target_width), PAD_ID, dtype=torch.long)
    decoder_mask = torch.zeros_like(decoder_ids, dtype=torch.bool)
    labels = torch.full_like(decoder_ids, -100)
    for i, r in enumerate(rows):
        source, target = r['source_ids'], r['target']
        encoder_ids[i, :len(source)] = torch.tensor(source)
        encoder_mask[i, :len(source)] = True
        decoder_ids[i, :len(target)] = torch.tensor([DECODER_START_ID] + target[:-1])
        decoder_mask[i, :len(target)] = True
        labels[i, :len(target)] = torch.tensor(target)
    return dict(encoder_ids=encoder_ids, encoder_mask=encoder_mask,
                decoder_ids=decoder_ids, decoder_mask=decoder_mask, labels=labels,
                doc_ids=[r['doc_id'] for r in rows])

datasets = {s: SummaryDataset(rows, s) for s, rows in data.items()}
def make_loader(split):
    return DataLoader(datasets[split], batch_size=BATCH_SIZE, shuffle=(split == 'train'),
                      collate_fn=collate_summary, num_workers=0, pin_memory=(DEVICE == 'cuda'))


Sentence selector epoch 1 (CPU):   0%|          | 0/6676 [00:00<?, ?it/s]

Sentence selector epoch 2 (CPU):   0%|          | 0/6676 [00:00<?, ?it/s]

Tokenizing train:   0%|          | 0/6676 [00:00<?, ?it/s]

train document tokens p50/p90/p99/max: [3675.0, 9134.5, 28179.5, 162919.0]
train summary including EOS tokens p50/p90/p99/max: [756.0, 1784.5, 5057.5, 32542.0]
train documents needing source selection: 6644
train reference summaries truncated for training/loss: 6403 / 6676


Tokenizing dev:   0%|          | 0/100 [00:00<?, ?it/s]

dev document tokens p50/p90/p99/max: [3808.0, 10870.9990234375, 36489.94921875, 105671.0]
dev summary including EOS tokens p50/p90/p99/max: [741.5, 1594.5997314453125, 8068.5966796875, 11097.0]
dev documents needing source selection: 100
dev reference summaries truncated for training/loss: 98 / 100


In [6]:
RMS_NORM_EPS = 1e-6
ROPE_BASE = 10000
DIM=640
ENCODER_LAYERS=6
DECODER_LAYERS=6
N_HEADS=16
N_KV_HEADS=4
FF_DIM=1728
MAX_ENCODER_LEN=512
MAX_DECODER_LEN=256
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# RMSNorm
# ============================================================

class RMSNorm(nn.Module):

    def __init__(self, dim, eps=RMS_NORM_EPS):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )

        self.eps = eps

    def forward(self, x):

        rms = (x*x).mean(
            dim=-1,
            keepdim=True
        )

        return self.weight * x * torch.rsqrt(
            rms + self.eps
        )


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):

    def __init__(
        self,
        head_dim,
        max_seq_len,
        base=ROPE_BASE,
    ):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2,
                    dtype=torch.float32,
                ) / head_dim
            )
        )

        positions = torch.arange(
            max_seq_len,
            dtype=torch.float32,
        )

        freqs = torch.outer(
            positions,
            inv_freq,
        )

        # [T, head_dim]
        emb = torch.cat(
            [freqs, freqs],
            dim=-1,
        )

        self.register_buffer(
            "cos",
            emb.cos()[None, None, :, :],
            persistent=False,
        )

        self.register_buffer(
            "sin",
            emb.sin()[None, None, :, :],
            persistent=False,
        )

    def forward(self, seq_len):

        return (
            self.cos[:, :, :seq_len],
            self.sin[:, :, :seq_len],
        )


def rotate_half(x):

    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]

    return torch.cat(
        [-x2, x1],
        dim=-1,
    )


def apply_rope(q, k, cos, sin):

    q = (
        q * cos
        +
        rotate_half(q) * sin
    )

    k = (
        k * cos
        +
        rotate_half(k) * sin
    )

    return q, k


# ============================================================
# SwiGLU
# ============================================================

class SwiGLU(nn.Module):

    def __init__(
        self,
        dim,
        hidden_dim,
    ):
        super().__init__()

        self.gate_proj = nn.Linear(
            dim,
            hidden_dim,
            bias=False,
        )

        self.up_proj = nn.Linear(
            dim,
            hidden_dim,
            bias=False,
        )

        self.down_proj = nn.Linear(
            hidden_dim,
            dim,
            bias=False,
        )

    def forward(self, x):

        return self.down_proj(
            F.silu(
                self.gate_proj(x)
            )
            *
            self.up_proj(x)
        )


# ============================================================
# GQA + Flash Attention
# ============================================================

class Attention(nn.Module):

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_heads,
        max_seq_len,
        causal=False,
        use_rope=True,
    ):
        super().__init__()

        assert dim % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.dim = dim
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads

        self.head_dim = dim // n_heads

        self.causal = causal
        self.use_rope = use_rope

        # ----------------------------------------------------
        # Q / K / V projections
        # ----------------------------------------------------

        self.q_proj = nn.Linear(
            dim,
            n_heads * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            dim,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            dim,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.o_proj = nn.Linear(
            n_heads * self.head_dim,
            dim,
            bias=False,
        )

        # RoPE only needs to be constructed for self-attention
        if use_rope:

            self.rope = RotaryEmbedding(
                self.head_dim,
                max_seq_len,
            )

    def forward(
        self,
        x,
        kv=None,
        past_kv=None,
        use_cache=False,
        key_mask=None,
    ):

        B, T, C = x.shape

        # ====================================================
        # Queries
        # ====================================================

        q = self.q_proj(x)

        q = q.view(
            B,
            T,
            self.n_heads,
            self.head_dim,
        )

        q = q.transpose(1, 2)

        # ====================================================
        # Keys / Values
        # ====================================================

        # Self-attention:
        #     source = x
        #
        # Cross-attention:
        #     source = encoder output

        source = x if kv is None else kv

        S = source.shape[1]

        k = self.k_proj(source)
        v = self.v_proj(source)

        k = k.view(
            B,
            S,
            self.n_kv_heads,
            self.head_dim,
        )

        v = v.view(
            B,
            S,
            self.n_kv_heads,
            self.head_dim,
        )

        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # ====================================================
        # RoPE
        # ====================================================

        # Only self-attention gets RoPE.
        # Cross-attention does NOT use RoPE.

        if kv is None and self.use_rope:

            current_len = k.shape[2]
            offset = 0 if past_kv is None else past_kv[0].shape[2]
            if offset + current_len > self.rope.cos.size(2):
                raise ValueError('Self-attention exceeds the Phase 1 positional context')
            cos, sin = self.rope(offset + current_len)
            cos = cos[:, :, offset:offset+current_len]
            sin = sin[:, :, offset:offset+current_len]

            q, k = apply_rope(
                q,
                k,
                cos,
                sin,
            )

        # ====================================================
        # Decoder KV Cache
        # ====================================================

        if (
            kv is None
            and past_kv is not None
        ):

            past_k, past_v = past_kv

            k = torch.cat(
                [past_k, k],
                dim=2,
            )

            v = torch.cat(
                [past_v, v],
                dim=2,
            )

        # Store native KV heads before expanding them for GQA.
        cache_kv = (k, v) if use_cache and kv is None else None

        # ====================================================
        # GQA
        # ====================================================

        if self.n_kv_heads != self.n_heads:

            repeat_factor = (
                self.n_heads
                // self.n_kv_heads
            )

            k = k.repeat_interleave(
                repeat_factor,
                dim=1,
            )

            v = v.repeat_interleave(
                repeat_factor,
                dim=1,
            )

        # ====================================================
        # Flash / SDPA Attention
        # ====================================================

        # During cached decoding, only one new token is
        # queried, so causal masking is unnecessary.

        is_causal = (
            self.causal
            and kv is None
            and past_kv is None
        )

        attention_mask = None
        if self.causal and kv is None and past_kv is not None:
            offset = past_kv[0].size(2)
            attention_mask = (torch.arange(k.size(2), device=x.device)[None, :] <=
                              (offset + torch.arange(T, device=x.device))[:, None])[None, None]
        if key_mask is not None:
            valid_keys = key_mask[:, None, None, :].bool()
            attention_mask = valid_keys if attention_mask is None else attention_mask & valid_keys
            if is_causal:
                causal_mask = torch.ones(T, k.size(2), device=x.device, dtype=torch.bool).tril()
                attention_mask = attention_mask & causal_mask[None, None]
            is_causal = False

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attention_mask,
            dropout_p=0.0,
            is_causal=is_causal,
        )

        # ====================================================
        # Output projection
        # ====================================================

        out = out.transpose(
            1,
            2,
        ).contiguous()

        out = out.view(
            B,
            T,
            C,
        )

        out = self.o_proj(out)

        # Cache only decoder self-attention
        new_kv = None

        if use_cache and kv is None:
            new_kv = cache_kv

        return out, new_kv


# ============================================================
# Encoder Block
# ============================================================

class EncoderBlock(nn.Module):

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_heads,
        ff_dim,
        max_seq_len,
    ):
        super().__init__()

        self.norm1 = RMSNorm(dim)

        self.self_attn = Attention(
            dim=dim,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            max_seq_len=max_seq_len,
            causal=False,
            use_rope=True,
        )

        self.norm2 = RMSNorm(dim)

        self.ffn = SwiGLU(
            dim,
            ff_dim,
        )

    def forward(self, x, key_mask=None):

        # Pre-norm self-attention
        x = x + self.self_attn(
            self.norm1(x), key_mask=key_mask
        )[0]

        # Pre-norm FFN
        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ============================================================
# Decoder Block
# ============================================================

class DecoderBlock(nn.Module):

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_heads,
        ff_dim,
        max_self_attn_len,
        max_cross_attn_len,
    ):
        super().__init__()

        # ----------------------------------------------------
        # Causal self-attention
        # ----------------------------------------------------

        self.norm1 = RMSNorm(dim)

        self.self_attn = Attention(
            dim=dim,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            max_seq_len=max_self_attn_len,
            causal=True,
            use_rope=True,
        )

        # ----------------------------------------------------
        # Encoder-decoder cross-attention
        # ----------------------------------------------------

        self.norm2 = RMSNorm(dim)

        self.cross_attn = Attention(
            dim=dim,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            max_seq_len=max_cross_attn_len,
            causal=False,
            use_rope=False,
        )

        # ----------------------------------------------------
        # FFN
        # ----------------------------------------------------

        self.norm3 = RMSNorm(dim)

        self.ffn = SwiGLU(
            dim,
            ff_dim,
        )

    def forward(
        self,
        x,
        encoder_output,
        past_kv=None,
        use_cache=False,
        key_mask=None,
        encoder_mask=None,
    ):

        # ====================================================
        # 1. Decoder causal self-attention
        # ====================================================

        residual = x

        x_norm = self.norm1(x)

        self_out, new_kv = self.self_attn(
            x_norm,
            past_kv=past_kv,
            use_cache=use_cache,
            key_mask=key_mask,
        )

        x = residual + self_out

        # ====================================================
        # 2. Cross-attention
        # ====================================================

        residual = x

        x_norm = self.norm2(x)

        cross_out, _ = self.cross_attn(
            x_norm,
            kv=encoder_output,
            key_mask=encoder_mask,
        )

        x = residual + cross_out

        # ====================================================
        # 3. FFN
        # ====================================================

        x = x + self.ffn(
            self.norm3(x)
        )

        return x, new_kv


# ============================================================
# Encoder-Decoder Language Model
# ============================================================

class EncoderDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        dim=DIM,
        encoder_layers=ENCODER_LAYERS,
        decoder_layers=DECODER_LAYERS,
        n_heads=N_HEADS,
        n_kv_heads=N_KV_HEADS,
        ff_dim=FF_DIM,
        max_encoder_len=MAX_ENCODER_LEN,
        max_decoder_len=MAX_DECODER_LEN,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim

        # ====================================================
        # Shared token embedding
        # ====================================================

        self.token_embedding = nn.Embedding(
            vocab_size,
            dim,
        )

        # ====================================================
        # Encoder
        # ====================================================

        self.encoder_blocks = nn.ModuleList([
            EncoderBlock(
                dim=dim,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                ff_dim=ff_dim,
                max_seq_len=max_encoder_len,
            )
            for _ in range(encoder_layers)
        ])

        self.encoder_norm = RMSNorm(dim)

        # ====================================================
        # Decoder
        # ====================================================

        self.decoder_blocks = nn.ModuleList([
            DecoderBlock(
                dim=dim,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                ff_dim=ff_dim,
                max_self_attn_len=max_decoder_len,
                max_cross_attn_len=max_encoder_len,
            )
            for _ in range(decoder_layers)
        ])

        self.decoder_norm = RMSNorm(dim)

        # ====================================================
        # Output head
        # ====================================================

        self.lm_head = nn.Linear(
            dim,
            vocab_size,
            bias=False,
        )

        # Tie output projection to token embedding
        self.lm_head.weight = (
            self.token_embedding.weight
        )

    # ========================================================
    # ENCODER
    # ========================================================

    def encode(self, encoder_ids):

        x = self.token_embedding(
            encoder_ids
        )

        for block in self.encoder_blocks:
            x = block(x)

        x = self.encoder_norm(x)

        return x

    # ========================================================
    # DECODER
    # ========================================================

    def decode(
        self,
        decoder_ids,
        encoder_output,
        past_kv=None,
        use_cache=False,
    ):

        x = self.token_embedding(
            decoder_ids
        )

        new_cache = []

        for i, block in enumerate(
            self.decoder_blocks
        ):

            layer_past = None

            if past_kv is not None:
                layer_past = past_kv[i]

            x, layer_cache = block(
                x=x,
                encoder_output=encoder_output,
                past_kv=layer_past,
                use_cache=use_cache,
            )

            if use_cache:
                new_cache.append(
                    layer_cache
                )

        x = self.decoder_norm(x)

        logits = self.lm_head(x)

        if use_cache:
            return logits, new_cache

        return logits, None

    # ========================================================
    # FORWARD
    # ========================================================

    def forward(
        self,
        encoder_ids,
        decoder_ids,
        labels=None,
    ):

        # Encode 512-token context
        encoder_output = self.encode(
            encoder_ids
        )

        # Decode continuation
        logits, _ = self.decode(
            decoder_ids,
            encoder_output,
        )

        # No labels -> inference
        if labels is None:
            return logits

        # ====================================================
        # Language-model loss
        # ====================================================

        loss = F.cross_entropy(
            logits.reshape(
                -1,
                self.vocab_size,
            ),
            labels.reshape(-1),
            ignore_index=-100,
        )

        return logits, loss

    # ========================================================
    # GENERATION
    # ========================================================

    @torch.no_grad()
    def generate(
        self,
        encoder_ids,
        bos_id,
        eos_id=None,
        max_new_tokens=128,
        temperature=1.0,
        top_p=0.9,
    ):

        self.eval()

        # ----------------------------------------------------
        # Encode context ONCE
        # ----------------------------------------------------

        encoder_output = self.encode(
            encoder_ids
        )

        # ----------------------------------------------------
        # Start decoder
        # ----------------------------------------------------

        B = encoder_ids.size(0)

        decoder_ids = torch.full(
            (B, 1),
            bos_id,
            dtype=torch.long,
            device=encoder_ids.device,
        )

        past_kv = None

        # ----------------------------------------------------
        # Autoregressive generation
        # ----------------------------------------------------

        for step in range(max_new_tokens):

            # Only feed the newest token once cache exists
            if past_kv is None:
                current_ids = decoder_ids
            else:
                current_ids = decoder_ids[:, -1:]

            logits, past_kv = self.decode(
                decoder_ids=current_ids,
                encoder_output=encoder_output,
                past_kv=past_kv,
                use_cache=True,
            )

            next_logits = logits[:, -1, :]

            # Temperature
            if temperature != 1.0:
                next_logits = (
                    next_logits / temperature
                )

            # ------------------------------------------------
            # Top-p sampling
            # ------------------------------------------------

            probs = F.softmax(
                next_logits,
                dim=-1,
            )

            sorted_probs, sorted_indices = torch.sort(
                probs,
                descending=True,
                dim=-1,
            )

            cumulative_probs = torch.cumsum(
                sorted_probs,
                dim=-1,
            )

            remove = (
                cumulative_probs > top_p
            )

            # Keep at least the first token
            remove[:, 0] = False

            sorted_probs[remove] = 0.0

            sorted_probs = (
                sorted_probs
                /
                sorted_probs.sum(
                    dim=-1,
                    keepdim=True,
                )
            )

            next_token = torch.multinomial(
                sorted_probs,
                num_samples=1,
            )

            next_token = torch.gather(
                sorted_indices,
                -1,
                next_token,
            )

            decoder_ids = torch.cat(
                [
                    decoder_ids,
                    next_token,
                ],
                dim=1,
            )

            if (
                eos_id is not None
                and torch.all(
                    next_token == eos_id
                )
            ):
                break

        # Remove BOS
        return decoder_ids[:, 1:]


# ============================================================
# Parameter count
# ============================================================




In [7]:
from torch.utils.checkpoint import checkpoint as activation_checkpoint
from contextlib import contextmanager

class SummaryModel(EncoderDecoderLM):
    def encode(self, encoder_ids, encoder_mask=None):
        x = self.token_embedding(encoder_ids)
        for block in self.encoder_blocks:
            if self.training and USE_GRADIENT_CHECKPOINTING:
                x = activation_checkpoint(block, x, key_mask=encoder_mask, use_reentrant=False,
                                          preserve_rng_state=True)
            else: x = block(x, key_mask=encoder_mask)
        return self.encoder_norm(x)

    def decode(self, decoder_ids, encoder_output, past_kv=None, use_cache=False,
               decoder_mask=None, encoder_mask=None):
        x = self.token_embedding(decoder_ids)
        new_cache = []
        for i, block in enumerate(self.decoder_blocks):
            layer_past = None if past_kv is None else past_kv[i]
            if self.training and USE_GRADIENT_CHECKPOINTING:
                if use_cache: raise ValueError('Training checkpointing does not use the inference cache')
                x, layer_cache = activation_checkpoint(block, x, encoder_output,
                    key_mask=decoder_mask, encoder_mask=encoder_mask, use_reentrant=False,
                    preserve_rng_state=True)
            else:
                x, layer_cache = block(x, encoder_output, past_kv=layer_past, use_cache=use_cache,
                                       key_mask=decoder_mask, encoder_mask=encoder_mask)
            if use_cache: new_cache.append(layer_cache)
        return self.lm_head(self.decoder_norm(x)), new_cache if use_cache else None

    def forward(self, encoder_ids, decoder_ids, encoder_mask=None, decoder_mask=None):
        memory = self.encode(encoder_ids, encoder_mask)
        logits, _ = self.decode(decoder_ids, memory, encoder_mask=encoder_mask,
                                decoder_mask=decoder_mask)
        return logits

phase1_path = download_bucket_file(MODEL_PATH)

def build_summary_model():
    model = SummaryModel(vocab_size=VOCAB_SIZE, **MODEL_CONFIG)
    checkpoint = torch.load(phase1_path, map_location='cpu', weights_only=False)
    if 'model_state_dict' in checkpoint: state = checkpoint['model_state_dict']
    elif 'model' in checkpoint: state = checkpoint['model']
    elif 'state_dict' in checkpoint: state = checkpoint['state_dict']
    elif all(isinstance(v, torch.Tensor) for v in checkpoint.values()): state = checkpoint
    else: raise ValueError(f'Cannot identify weights: {list(checkpoint)}')
    # Restore the original vocabulary head too; no Phase 5 classification weights.
    state = dict(state)
    if 'lm_head.weight' not in state and 'token_embedding.weight' in state:
        state['lm_head.weight'] = state['token_embedding.weight']
    if not torch.equal(state['lm_head.weight'], state['token_embedding.weight']):
        raise ValueError('Checkpoint LM head and embedding disagree with the tied Phase 1 architecture')
    model.load_state_dict(state, strict=True)
    del state, checkpoint
    for p in model.parameters(): p.requires_grad = False
    # Fine-tune complete upper blocks directly; embeddings and tied head stay frozen.
    if UNFREEZE_ENCODER_BLOCKS:
        for block in model.encoder_blocks[-UNFREEZE_ENCODER_BLOCKS:]:
            block.requires_grad_(True)
    if UNFREEZE_DECODER_BLOCKS:
        for block in model.decoder_blocks[-UNFREEZE_DECODER_BLOCKS:]:
            block.requires_grad_(True)
    if TRAIN_FINAL_NORMS:
        model.encoder_norm.requires_grad_(True)
        model.decoder_norm.requires_grad_(True)
    model = model.to(DEVICE)
    assert model.lm_head.weight is model.token_embedding.weight
    print('Total/trainable parameters:', sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad))
    return model

def summary_loss_sum(model, batch):
    encoder_ids = batch['encoder_ids'].to(DEVICE)
    decoder_ids = batch['decoder_ids'].to(DEVICE)
    encoder_mask = batch['encoder_mask'].to(DEVICE)
    decoder_mask = batch['decoder_mask'].to(DEVICE)
    labels = batch['labels'].to(DEVICE)
    with autocast_context():
        logits = model(encoder_ids, decoder_ids, encoder_mask, decoder_mask)
    # Collation already shifts decoder inputs; do not shift these labels again.
    loss = F.cross_entropy(logits.float().reshape(-1,VOCAB_SIZE), labels.reshape(-1),
                           ignore_index=-100, reduction='sum')
    return loss, int((labels != -100).sum())

@torch.no_grad()
def generate_summary(model, source_ids):
    model.eval()
    encoder_ids = torch.tensor([source_ids], device=DEVICE, dtype=torch.long)
    encoder_mask = torch.ones_like(encoder_ids, dtype=torch.bool)
    with autocast_context(): memory = model.encode(encoder_ids, encoder_mask)
    ids = torch.tensor([[DECODER_START_ID]], device=DEVICE, dtype=torch.long)
    past, generated = None, []
    for _ in range(MAX_NEW_TOKENS):
        with autocast_context():
            logits, past = model.decode(ids, memory, past_kv=past, use_cache=True,
                                         encoder_mask=encoder_mask)
        next_id = int(logits[0,-1].argmax())
        if next_id == EOS_ID: break
        generated.append(next_id)
        ids = torch.tensor([[next_id]], device=DEVICE, dtype=torch.long)
    return tokenizer.decode(generated, skip_special_tokens=True).strip(), len(generated)

@contextmanager
def evaluation_weights(model):
    # Direct partial fine-tuning has no low-rank matrices to merge.
    if model.training: raise RuntimeError('Evaluation context requires model.eval()')
    yield

@torch.no_grad()
def check_generation_cache(model):
    model.eval()
    # Compare cached decoding to teacher forcing, including absolute RoPE offsets.
    row = datasets['train'].rows[0]
    source = torch.tensor([row['source_ids'][:12]], device=DEVICE)
    target = [DECODER_START_ID] + row['target'][:2]
    decoder = torch.tensor([target], device=DEVICE)
    with evaluation_weights(model):
        memory = model.encode(source)
        expected, _ = model.decode(decoder, memory)
        past = None
        for i in range(decoder.size(1)):
            actual, past = model.decode(decoder[:,i:i+1], memory, past_kv=past, use_cache=True)
            torch.testing.assert_close(actual, expected[:,i:i+1], rtol=2e-4, atol=2e-4)
    print('Cached/full decoder logits agree on the startup check')

# Training-time model selection still uses the baseline greedy decoder. After
# choosing the checkpoint, tune these fixed candidates on held-out documents.
DECODE_GRID = [dict(name='greedy',beam=1,alpha=0.,no_repeat=0),
               dict(name='beam2',beam=2,alpha=0.6,no_repeat=0),
               dict(name='beam4',beam=4,alpha=0.6,no_repeat=0),
               dict(name='beam4_repeat',beam=4,alpha=0.6,no_repeat=3)]
DECODE_TUNE_DOCS = 50
BEST_DECODE = DECODE_GRID[0]

def repeated_ngram_tokens(ids,n):
    if n<=0 or len(ids)<n-1:return set()
    prefix=tuple(ids[-(n-1):]) if n>1 else ()
    return {ids[i+n-1] for i in range(len(ids)-n+1) if tuple(ids[i:i+n-1])==prefix}

@torch.no_grad()
def generate_with_settings(model,source_ids,settings):
    if settings['beam']==1:return generate_summary(model,source_ids)
    model.eval()
    source=torch.tensor([source_ids],device=DEVICE)
    mask=torch.ones_like(source,dtype=torch.bool)
    with autocast_context():memory=model.encode(source,mask)
    # Live beam: generated IDs, cumulative raw log-probability, parent KV cache.
    live=[([],0.,None)]
    finished=[]
    def rank(item):
        ids,score=item[:2]
        return score/(((5+len(ids))/6)**settings['alpha'])
    for step in range(MAX_NEW_TOKENS):
        candidates=[]
        for ids,score,past in live:
            token=ids[-1] if ids else DECODER_START_ID
            with autocast_context():
                logits,cache=model.decode(torch.tensor([[token]],device=DEVICE),memory,
                    past_kv=past,use_cache=True,encoder_mask=mask)
            logp=F.log_softmax(logits[0,-1].float(),dim=-1)
            banned=repeated_ngram_tokens(ids,settings['no_repeat'])
            if banned:logp[list(banned)]=-float('inf')
            # EOS is always considered, even outside the beam's token top-k.
            eos_score=float(logp[EOS_ID])
            if math.isfinite(eos_score):finished.append((ids+[EOS_ID],score+eos_score))
            logp[EOS_ID]=-float('inf')
            values,indices=torch.topk(logp,min(settings['beam'],logp.numel()-1))
            for value,index in zip(values.tolist(),indices.tolist()):
                if math.isfinite(value):candidates.append((ids+[index],score+value,cache))
        finished=sorted(finished,key=rank,reverse=True)[:settings['beam']]
        live=sorted(candidates,key=rank,reverse=True)[:settings['beam']]
        if not live:break
    # Include length-capped unfinished hypotheses; no forced EOS replaces content.
    best=max(finished+[(ids,score) for ids,score,_ in live],key=rank)
    ids=best[0]
    if ids and ids[-1]==EOS_ID:ids=ids[:-1]
    return tokenizer.decode(ids,skip_special_tokens=True).strip(),len(ids)

@torch.no_grad()
def tune_decoding(model):
    scorer=rouge_scorer.RougeScorer(['rouge1','rougeL'],use_stemmer=True)
    rows=datasets['dev'].rows[:DECODE_TUNE_DOCS]
    results=[]
    model.eval()
    with evaluation_weights(model):
        for setting in DECODE_GRID:
            scores1,scoresL,lengths,repeats=[],[],[],[]
            start=time.perf_counter()
            for row in tqdm(rows,desc='Tune '+setting['name']):
                text,length=generate_with_settings(model,row['source_ids'],setting)
                score=scorer.score(row['reference'],text)
                scores1.append(score['rouge1'].fmeasure);scoresL.append(score['rougeL'].fmeasure)
                lengths.append(length)
                words=text.lower().split();grams=list(zip(words,words[1:],words[2:]))
                repeats.append(1-len(set(grams))/max(1,len(grams)) if grams else 0.)
            result=dict(**setting,rouge1_f1=float(np.mean(scores1)),rougeL_f1=float(np.mean(scoresL)),
                        mean_generated_tokens=float(np.mean(lengths)),trigram_repeat_fraction=float(np.mean(repeats)),
                        fraction_at_limit=float(np.mean(np.array(lengths)>=MAX_NEW_TOKENS)),
                        seconds=time.perf_counter()-start)
            print(result);results.append(result)
    best=max(results,key=lambda r:r['rougeL_f1'])
    setting=next(s for s in DECODE_GRID if s['name']==best['name'])
    report=dict(selected=setting,selection_metric='rougeL_f1',docs=len(rows),results=results,
                mean_full_reference_tokens=float(np.mean([len(tokenizer.encode(r['reference'],add_special_tokens=False).ids) for r in rows])),
                note='Reuses model-selection dev split; these are tuning results, not an independent evaluation.')
    with open(WORK_DIR/'decoding_selection.json','w') as f:json.dump(report,f,indent=2)
    if UPLOAD_CHECKPOINTS:
        fs.put_file(str(WORK_DIR/'decoding_selection.json'),bucket_path(f'{CHECKPOINT_PREFIX}/{BEST_TAG}/decoding_selection.json'))
    return setting


(…)eckpoints/encoder_decoder_mixed/model.pt:   0%|          | 0.00/318M [00:00<?, ?B/s]

In [8]:
def experiment_metadata(domain='SUMM'):
    return dict(domain=domain, phase1_tag=PHASE1_TAG, model_path=MODEL_PATH, tokenizer_path=TOKENIZER_PATH,
                model_config=MODEL_CONFIG,
                max_seq_len=MAX_SEQ_LEN, max_new_tokens=MAX_NEW_TOKENS, source_budget=SOURCE_BUDGET,
                source_spans=SOURCE_SPANS, source_selection='learned_sentence_selector_v1', selector_config=SELECTOR_CONFIG, selector_hash=SELECTOR_HASH, selector_sklearn=sklearn.__version__, decoder_start_id=DECODER_START_ID, attention='bidirectional_encoder_causal_decoder',
                unfreeze_encoder_blocks=UNFREEZE_ENCODER_BLOCKS,
                unfreeze_decoder_blocks=UNFREEZE_DECODER_BLOCKS, train_final_norms=TRAIN_FINAL_NORMS, gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
                use_amp=USE_AMP, amp_dtype=str(AMP_DTYPE), data_repo=DATA_REPO, data_prefix=DATA_PREFIX,
                data_fingerprint=DATA_FINGERPRINT, val_fraction=VAL_FRACTION, val_max_docs=VAL_MAX_DOCS,
                technique=TECHNIQUE, train_fraction=TRAIN_FRACTION,
                seed=SEED, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
                lr=LR, weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP, patience=PATIENCE, min_delta=MIN_DELTA,
                scheduler=dict(name='warmup_then_validation_plateau', warmup_ratio=WARMUP_RATIO,
                               start_factor=WARMUP_START_FACTOR, factor=LR_FACTOR, bad_checks=LR_BAD_CHECKS,
                               cooldown_checks=LR_COOLDOWN_CHECKS, min_lr=LR_MIN, loss_threshold=LR_LOSS_THRESHOLD,
                               checks_per_epoch=VAL_LOSS_CHECKS_PER_EPOCH, planned_epochs=NUM_EPOCHS),
                summary_key=SUMMARY_KEY, skip_empty_summaries=SKIP_EMPTY_SUMMARIES,
                selection_metric='rougeL_f1', decoding='greedy', loss='encoder_decoder_summary_token_mean')

def save_summary_checkpoint(tag, model, optimizer, epoch, extra, domain):
    folder = WORK_DIR / 'checkpoints' / tag
    folder.mkdir(parents=True, exist_ok=True)
    rng = {'python': random.getstate(), 'torch': torch.get_rng_state()}
    if torch.cuda.is_available(): rng['cuda'] = torch.cuda.get_rng_state_all()
    payload = dict(model=model.state_dict(), optimizer=optimizer.state_dict(), rng=rng,
                   epoch=epoch, extra=extra, metadata=experiment_metadata(domain), sentence_selector=SELECTOR_PAYLOAD)
    tmp = folder / 'checkpoint.tmp'
    torch.save(payload, tmp)
    os.replace(tmp, folder / 'checkpoint.pt')
    if UPLOAD_CHECKPOINTS:
        fs.put_file(str(folder / 'checkpoint.pt'), checkpoint_remote_path(tag))

def load_summary_checkpoint(tag, model, domain, optimizer=None, restore_rng=False):
    path = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
    if not path.exists() or restore_rng:
        if UPLOAD_CHECKPOINTS:
            path = download_bucket_file(checkpoint_remote_path(tag))
        elif not path.exists():
            raise FileNotFoundError(path)
    payload = torch.load(path, map_location='cpu', weights_only=False)  # own trusted checkpoints only
    assert payload['metadata'] == experiment_metadata(domain), 'Checkpoint configuration mismatch'
    model.load_state_dict(payload['model'])
    if optimizer is not None: optimizer.load_state_dict(payload['optimizer'])
    if restore_rng:
        random.setstate(payload['rng']['python'])
        torch.set_rng_state(payload['rng']['torch'])
        if torch.cuda.is_available() and 'cuda' in payload['rng']:
            torch.cuda.set_rng_state_all(payload['rng']['cuda'])
    return payload


class ValidationLRScheduler:
    def __init__(self, optimizer, total_updates):
        assert total_updates > 0 and 0 < WARMUP_RATIO < 1
        assert 0 < WARMUP_START_FACTOR <= 1 and 0 < LR_FACTOR < 1
        assert LR_BAD_CHECKS >= 1 and 0 <= LR_MIN <= LR
        self.optimizer = optimizer
        self.updates = 0
        self.warmup_updates = max(1, math.ceil(total_updates * WARMUP_RATIO))
        self.peak_lrs = [g['lr'] for g in optimizer.param_groups]
        # PyTorch reduces after num_bad_epochs > patience: 1 means two bad checks.
        self.plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=LR_FACTOR, patience=LR_BAD_CHECKS-1,
            threshold=LR_LOSS_THRESHOLD, threshold_mode='rel',
            cooldown=LR_COOLDOWN_CHECKS, min_lr=LR_MIN)
        self._set_warmup_lr()

    def _set_warmup_lr(self):
        factor = WARMUP_START_FACTOR + (1-WARMUP_START_FACTOR) * self.updates/self.warmup_updates
        for group, peak in zip(self.optimizer.param_groups, self.peak_lrs):
            group['lr'] = peak * factor

    def optimizer_step_completed(self):
        self.updates += 1
        if self.updates <= self.warmup_updates: self._set_warmup_lr()

    def validation_step(self, loss):
        if not math.isfinite(loss): raise ValueError('Non-finite validation loss')
        # Warmup and plateau never fight over the learning rate.
        if self.updates >= self.warmup_updates: self.plateau.step(loss)

    def state_dict(self):
        return dict(updates=self.updates, warmup_updates=self.warmup_updates,
                    peak_lrs=self.peak_lrs, plateau=self.plateau.state_dict(),
                    current_lrs=[g['lr'] for g in self.optimizer.param_groups])

    def load_state_dict(self, state):
        if state['warmup_updates'] != self.warmup_updates:
            raise ValueError('Warmup horizon changed; resume with the original dataset, epochs and accumulation')
        self.updates = state['updates']
        self.peak_lrs = state['peak_lrs']
        self.plateau.load_state_dict(state['plateau'])
        for group, lr in zip(self.optimizer.param_groups, state['current_lrs']): group['lr'] = lr

@torch.no_grad()
def evaluate_summary_loss(model, loader):
    was_training = model.training
    model.eval()
    try:
        total_loss = total_tokens = 0
        for batch in loader:
            loss, count = summary_loss_sum(model, batch)
            total_loss += loss.item(); total_tokens += count
        if total_tokens == 0: raise ValueError('No validation summary targets')
        return total_loss/total_tokens
    finally:
        model.train(was_training)

@torch.no_grad()
def evaluate_summary(model, loader):
    model.eval()
    validation_loss = evaluate_summary_loss(model, loader)
    scorer = rouge_scorer.RougeScorer(['rouge1','rougeL'],use_stemmer=True)
    records = []
    started = time.perf_counter()
    with evaluation_weights(model):
        for row in tqdm(datasets['dev'].rows,desc='Generate validation summaries'):
            prediction, tokens = generate_summary(model,row['source_ids'])
            scores = scorer.score(row['reference'],prediction)
            words = prediction.lower().split()
            trigrams = list(zip(words,words[1:],words[2:]))
            records.append(dict(id=row['doc_id'],summary=prediction,reference=row['reference'],
                                rouge1_f1=scores['rouge1'].fmeasure,rougeL_f1=scores['rougeL'].fmeasure,
                                generated_tokens=tokens,repeated_trigram_fraction=1-len(set(trigrams))/len(trigrams) if trigrams else 0.0))
    metrics = {key:sum(r[key] for r in records)/len(records) for key in ['rouge1_f1','rougeL_f1','generated_tokens','repeated_trigram_fraction']}
    metrics.update(loss=validation_loss, generation_seconds=time.perf_counter()-started)
    with open(WORK_DIR / 'SUMM_dev_predictions.json','w',encoding='utf-8') as f:
        json.dump(records,f,ensure_ascii=False,indent=2)
    return metrics

def cleanup_latest_checkpoint():
    # Exact file deletion only. Never delete a parent directory or a Phase-1 file.
    best_local = WORK_DIR / 'checkpoints' / BEST_TAG / 'checkpoint.pt'
    if not best_local.is_file(): raise RuntimeError('Best checkpoint is missing; keeping latest')
    if UPLOAD_CHECKPOINTS:
        best_remote = checkpoint_remote_path(BEST_TAG)
        if not fs.exists(best_remote) or fs.info(best_remote)['size'] != best_local.stat().st_size:
            raise RuntimeError('Best upload not verified; keeping latest')
        latest_remote = checkpoint_remote_path(SAVE_TAG)
        assert latest_remote != best_remote and latest_remote != bucket_path(MODEL_PATH)
        if fs.exists(latest_remote): fs.rm(latest_remote)
        if fs.exists(latest_remote): raise RuntimeError('Latest bucket checkpoint deletion failed')
    latest_local = WORK_DIR / 'checkpoints' / SAVE_TAG / 'checkpoint.pt'
    if latest_local.exists(): latest_local.unlink()
    print('Training completed: retained best checkpoint only:', BEST_TAG)


In [9]:
def train_summary(domain='SUMM'):
    assert domain == 'SUMM'
    assert_run_destination()
    seed_everything(SEED)
    model = build_summary_model()
    check_generation_cache(model)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and AMP_DTYPE == torch.float16)
    tag = SAVE_TAG
    best_tag = BEST_TAG
    train_loader, dev_loader = make_loader('train'), make_loader('dev')
    assert len(train_loader) > 0 and VAL_LOSS_CHECKS_PER_EPOCH >= 1
    updates_per_epoch = math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)
    check_interval = max(1, math.ceil(updates_per_epoch/VAL_LOSS_CHECKS_PER_EPOCH))
    scheduler = ValidationLRScheduler(optimizer, NUM_EPOCHS * updates_per_epoch)
    print('Optimizer updates per epoch:', updates_per_epoch, '| warmup updates:', scheduler.warmup_updates)
    start = 0
    extra = dict(best_f1=-1.0, patience_reference=-1.0, stale=0, history=[], best_tag=best_tag)
    if RESUME:
        checkpoint = load_summary_checkpoint(SAVE_TAG, model, domain, optimizer, restore_rng=True)
        start, extra = checkpoint['epoch'], checkpoint['extra']
        best_tag = extra['best_tag']
        if 'scaler' in extra: scaler.load_state_dict(extra['scaler'])
        if 'scheduler' not in extra: raise ValueError('This checkpoint predates the scheduler; start a new run tag')
        scheduler.load_state_dict(extra['scheduler'])
        del checkpoint
    for epoch in range(start, NUM_EPOCHS):
        if extra['stale'] >= PATIENCE: break
        model.train()
        optimizer.zero_grad(set_to_none=True)
        train_sum = train_n = group_n = seen_docs = 0
        next_log = LOG_EVERY_DOCS
        epoch_update = 0
        for step, batch in enumerate(tqdm(train_loader, desc=f'{domain} epoch {epoch+1}')):
            loss_sum, token_count = summary_loss_sum(model, batch)
            scaler.scale(loss_sum).backward()
            group_n += token_count
            train_n += token_count
            train_sum += loss_sum.item()
            seen_docs += len(batch['doc_ids'])
            if seen_docs >= next_log:
                print(f'Epoch {epoch+1} | docs {seen_docs:,}/{len(datasets["train"]):,} | summary token loss {train_sum/train_n:.5f}')
                next_log += LOG_EVERY_DOCS
            if (step + 1) % GRAD_ACCUM_STEPS == 0 or step + 1 == len(train_loader):
                scaler.unscale_(optimizer)
                for p in params:
                    if p.grad is not None: p.grad.div_(group_n)
                torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
                previous_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                update_succeeded = scaler.get_scale() >= previous_scale
                if update_succeeded: scheduler.optimizer_step_completed()
                optimizer.zero_grad(set_to_none=True)
                group_n = 0
                epoch_update += 1
                if epoch_update % check_interval == 0 or step + 1 == len(train_loader):
                    dev_loss = evaluate_summary_loss(model, dev_loader)
                    old_lr = optimizer.param_groups[0]['lr']
                    scheduler.validation_step(dev_loss)
                    current_lr = optimizer.param_groups[0]['lr']
                    event = dict(epoch=epoch+1, optimizer_updates=scheduler.updates,
                                 dev_loss=dev_loss, lr_before=old_lr, lr=current_lr)
                    extra.setdefault('lr_history', []).append(event)
                    print(f"Validation | update {scheduler.updates:,} | summary loss {dev_loss:.5f} | LR {old_lr:.3g} -> {current_lr:.3g}")
        metrics = evaluate_summary(model, dev_loader)
        f1 = metrics['rougeL_f1']
        improved = f1 > extra['best_f1']
        if improved:
            extra['best_f1'] = f1
        if f1 > extra['patience_reference'] + MIN_DELTA:
            extra['patience_reference'], extra['stale'] = f1, 0
        else: extra['stale'] += 1
        extra['history'].append(dict(epoch=epoch+1, train_loss=train_sum/train_n, lr=optimizer.param_groups[0]['lr'], **metrics))
        print(f"SUMM train loss={train_sum/train_n:.4f}, dev ROUGE-L F1={f1:.4f}, ROUGE-1 F1={metrics['rouge1_f1']:.4f}, patience={extra['stale']}/{PATIENCE}")
        extra['scaler'] = scaler.state_dict()
        extra['scheduler'] = scheduler.state_dict()
        if improved: save_summary_checkpoint(best_tag, model, optimizer, epoch+1, extra, domain)
        save_summary_checkpoint(tag, model, optimizer, epoch+1, extra, domain)
    best_payload = load_summary_checkpoint(best_tag, model, domain)
    del best_payload
    metrics = evaluate_summary(model, dev_loader)
    with open(WORK_DIR / f'{domain}_dev_metrics.json', 'w') as f: json.dump(metrics, f, indent=2)
    with open(WORK_DIR / f'{domain}_history.json', 'w') as f: json.dump(extra['history'], f, indent=2)
    with open(WORK_DIR / 'SUMM_lr_history.json', 'w') as f: json.dump(extra.get('lr_history', []), f, indent=2)
    print('Best development metrics:', metrics)
    cleanup_latest_checkpoint()
    del optimizer
    return model, extra['history']


In [10]:
def validate_predictions(predictions, documents):
    assert len(predictions) == len(documents)
    assert [p['id'] for p in predictions] == [d['id'] for d in documents]
    assert len({p['id'] for p in predictions}) == len(predictions)
    assert all(set(p) == {'id','summary'} and isinstance(p['summary'],str) for p in predictions)

@torch.no_grad()
def predict_summary(model):
    model.eval()
    predictions = []
    with evaluation_weights(model):
        for row in tqdm(datasets['test'].rows,desc='SUMM test predictions'):
            summary, _ = generate_with_settings(model,row['source_ids'],BEST_DECODE)
            predictions.append(dict(id=row['doc_id'],summary=summary))
    validate_predictions(predictions,data['test'])
    return predictions

summ_model, summ_history = train_summary()
BEST_DECODE = tune_decoding(summ_model)
# Load test only after training; test text is never used for fitting/selection.
data['test'] = normalize_documents(download_json(f'{DATA_PREFIX}/Test/test.json'),labelled=False)
datasets['test'] = SummaryDataset(data['test'],'test')
predictions = predict_summary(summ_model)
json_path = WORK_DIR / 'prediction.json'
with open(json_path,'w',encoding='utf-8') as f:
    json.dump(predictions,f,ensure_ascii=False,indent=2)
del summ_model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


Total/trainable parameters: 79499520 19422720
Cached/full decoder logits agree on the startup check
Optimizer updates per epoch: 835 | warmup updates: 209


SUMM epoch 1:   0%|          | 0/6676 [00:00<?, ?it/s]

Epoch 1 | docs 1,000/6,676 | summary token loss 4.96833
Validation | update 209 | summary loss 4.99701 | LR 2e-05 -> 2e-05
Epoch 1 | docs 2,000/6,676 | summary token loss 4.95766
Epoch 1 | docs 3,000/6,676 | summary token loss 4.93794
Validation | update 418 | summary loss 4.93859 | LR 2e-05 -> 2e-05
Epoch 1 | docs 4,000/6,676 | summary token loss 4.91980
Epoch 1 | docs 5,000/6,676 | summary token loss 4.90754
Validation | update 627 | summary loss 4.89554 | LR 2e-05 -> 2e-05
Epoch 1 | docs 6,000/6,676 | summary token loss 4.89737
Validation | update 835 | summary loss 4.86736 | LR 2e-05 -> 2e-05


Generate validation summaries:   0%|          | 0/100 [00:00<?, ?it/s]

SUMM train loss=4.8892, dev ROUGE-L F1=0.1329, ROUGE-1 F1=0.1733, patience=0/10


Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (45 outputs left out)

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

Tokenizing test:   0%|          | 0/100 [00:00<?, ?it/s]

test document tokens p50/p90/p99/max: [3800.5, 10392.498046875, 34018.77734375, 45874.0]
test documents needing source selection: 100


SUMM test predictions:   0%|          | 0/100 [00:00<?, ?it/s]

In [11]:
# Packaging can be rerun without retraining.
json_path = WORK_DIR / 'prediction.json'
with open(json_path, encoding='utf-8') as f: predictions = json.load(f)
validate_predictions(predictions, data['test'])
zip_path = WORK_DIR / SUBMISSION_ZIP_NAME
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(json_path, arcname='prediction.json')
print('JSON:', json_path)
print('ZIP:', zip_path)
print('Cases:', len(predictions))


JSON: /kaggle/working/summ_runs/summ_encoder_decoder_mixed_partial_ft_sentence_select_enc2_dec2_len512_summary256_selected_warmup_plateau/prediction.json
ZIP: /kaggle/working/summ_runs/summ_encoder_decoder_mixed_partial_ft_sentence_select_enc2_dec2_len512_summary256_selected_warmup_plateau/summ_encoder_decoder_mixed_partial_ft_sentence_select_enc2_dec2_len512_summary256_selected_warmup_plateau.zip
Cases: 100
